In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import xgboost as xgb
from sklearn.metrics import mean_absolute_error,mean_squared_error
import warnings

In [5]:
warnings.filterwarnings('ignore')

file_path = '../Data/demand_forecast_cleaned.csv'
df = pd.read_csv(file_path)

In [6]:
df

,Date,Product_Category,Demand_Capped,Demand_Smoothed
0,2012-01-01,Category_001,356.0,356.0
1,2012-01-08,Category_001,2163.0,2163.0
2,2012-01-15,Category_001,3903.0,3903.0
3,2012-01-22,Category_001,5120.0,5120.0
4,2012-01-29,Category_001,4176.0,4176.0
...,...,...,...,...
7989,2016-11-27,Category_033,150000.0,150000.0
7990,2016-12-04,Category_033,200000.0,200000.0
7991,2016-12-11,Category_033,120000.0,120000.0
7992,2016-12-18,Category_033,360000.0,360000.0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7994 entries, 0 to 7993
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              7994 non-null   object 
 1   Product_Category  7994 non-null   object 
 2   Demand_Capped     7994 non-null   float64
 3   Demand_Smoothed   7994 non-null   float64
dtypes: float64(2), object(2)
memory usage: 249.9+ KB


In [8]:
df['Date'] = pd.to_datetime(df['Date'])
display(df.head(5))

,Date,Product_Category,Demand_Capped,Demand_Smoothed
0,2012-01-01,Category_001,356.0,356.0
1,2012-01-08,Category_001,2163.0,2163.0
2,2012-01-15,Category_001,3903.0,3903.0
3,2012-01-22,Category_001,5120.0,5120.0
4,2012-01-29,Category_001,4176.0,4176.0


In [ ]:
results = []

categories = df['Product_Category'].unique()

print("เริ่มทำการประมวลผลเทรนโมเดล แข่งขันกันระหว่าง Prophet และ XGBoost...")

for cat in categories:
    print(f"กำลังประมวลผล: {cat}...")

    df_cat = df[df['Product_Category'] == cat].sort_values('Date').copy()

    #prepare data for feature engineering
    df_prophet = df_cat[['Date','Demand_Smoothed']].rename(columns={'Date':'ds', 'Demand_Smoothed':'y'})

    #XGBoost 
    df_xgb = df_cat[['Date', 'Demand_Smoothed']].copy()